# Задача 2.  Оптимизация скидки для удвоения продаж

**Цель:** На основе статистики цен и объема продаж по позициям дать предложение по размеру скидки, чтобы увеличить продажи в 2 раза

**Методология:** 
Комбинация эмпирического анализа и расчета индивидуальной ценовой эластичности спроса через OLS-регрессию.

## Аналитическая записка

При скидке 25% средние дневные продажи всех 10 товаров были в 2,2–3,3 раза выше базы без скидки. Это минимальная наблюдавшаяся скидка, при которой все позиции достигли удвоения продаж.

OLS-модель предлагает скидки 17,1–23,8%, но при таких скидках фактический рост продаж составлял лишь 1,1–1,7× (по 8 товарам, где есть наблюдения ниже 25%). Рост идёт ступенькой между 20% и 25%, лог-линейная регрессия её сглаживает, занижая скидку.

**Предложение:** проверить скидку 25% по всем позициям в пилоте с сопоставимым контролем. До постоянного внедрения подтвердить рост продаж и оценить прибыльность: историческое сравнение не учитывает календарные и другие сопутствующие факторы.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf


file_path = "data/Тестовое_задание_Data_аналитик_ЦО_2025_2.xlsx"
df = pd.read_excel(file_path, sheet_name="Данные для задачи 2")


display(df.head(3))

,Дата,Код товара,Количество продано,Розничная цена,Цена с учётом скидки
0,2024-06-01,3226300004,9,271,271.0
1,2024-06-02,3226300004,15,271,271.0
2,2024-06-03,3226300004,12,271,216.8


In [2]:
# базовые проверки
print(f'Пропуски в данных: {df.isna().sum().sum()}')
print(f'Продажи <= 0: {(df["Количество продано"] <= 0).sum()}')
print(f'Дубликатов (дата + товар): {df.duplicated(subset=["Дата", "Код товара"]).sum()}')

Пропуски в данных: 0
Продажи <= 0: 0
Дубликатов (дата + товар): 0


In [3]:
# подход 1: эмпирический, по каждому товару


#расчитываем рост продаж со скидкой к базе

df["Скидка_%"] = ((1 - df["Цена с учётом скидки"] / df["Розничная цена"]) * 100).round(2)

base_sales = df[df["Скидка_%"] == 0].groupby("Код товара")["Количество продано"].mean()

sales_by_discount = df[df["Скидка_%"] > 0].groupby(["Код товара", "Скидка_%"]).agg(
    Средние_продажи=("Количество продано", "mean"),
    Дней=("Количество продано", "count")
).reset_index().round(2)

sales_by_discount['Рост'] = (sales_by_discount["Средние_продажи"] / sales_by_discount["Код товара"].map(base_sales)).round(1)
 
print(f'Товаров, где достигнут рост x2 за счет промо: {sales_by_discount[sales_by_discount["Рост"] >= 2]["Код товара"].nunique()} из {sales_by_discount["Код товара"].nunique()}')     
print("\nЭмпирический рост продаж по товарам при разных скидках")

display(sales_by_discount[sales_by_discount["Рост"] >= 2]
    .style.format({
        "Скидка_%": "{:,.0f}%",
        "Средние_продажи": "{:,.2f}",
        "Дней": "{:,.0f}",
        "Рост": "{:,.1f}x"
    })
)

Товаров, где достигнут рост x2 за счет промо: 10 из 10

Эмпирический рост продаж по товарам при разных скидках


,Код товара,Скидка_%,Средние_продажи,Дней,Рост
1,3226300004,25%,23.38,8,2.3x
2,3226300004,30%,24.12,8,2.4x
3,3226300004,40%,37.00,4,3.6x
7,3234800007,25%,24.25,4,2.4x
8,3234800007,30%,25.75,4,2.5x
9,3234800007,35%,36.50,4,3.5x
10,14011400047,25%,22.00,4,2.2x
11,14011400047,30%,24.50,8,2.4x
12,14011400047,35%,33.50,8,3.3x
13,14011400047,40%,34.25,4,3.4x


In [4]:
# скидка для x2 по каждому товару

x2 = sales_by_discount[sales_by_discount["Рост"] >= 2]

result = (x2.sort_values(["Скидка_%"]).drop_duplicates("Код товара"))


print("Рекомендуемая скидка по товарам (по факту)")

display(result.style.format({
    "Скидка_%": "{:,.0f}%",
    "Средние_продажи": "{:,.0f}",
    "Рост": "{:,.1f}x"
}))

Рекомендуемая скидка по товарам (по факту)


,Код товара,Скидка_%,Средние_продажи,Дней,Рост
1,3226300004,25%,23,8,2.3x
37,19000013047,25%,25,8,2.5x
34,19000000363,25%,26,4,2.5x
30,15111300017,25%,16,8,3.3x
24,15111300006,25%,26,4,2.7x
43,19760336592,25%,26,8,2.6x
15,14012900005,25%,26,4,2.8x
20,15110900016,25%,23,8,2.3x
10,14011400047,25%,22,4,2.2x
7,3234800007,25%,24,4,2.4x


In [5]:
# подход 2: OLS-регрессия для проверки эмпирики

df_reg = df.rename(columns={
    "Цена с учётом скидки": "price",
    "Количество продано": "qty",
    "Код товара": "item_id"
})

results = []

for item in df_reg["item_id"].unique():
    sub_df = df_reg[df_reg["item_id"] == item]

    model = smf.ols("np.log(qty) ~ np.log(price)", data=sub_df).fit()

    elasticity = model.params["np.log(price)"]
    r_squared = model.rsquared
    discount_to_x2 = 1 - (2 ** (1 / elasticity)) if elasticity < 0 else np.nan

    results.append({
        "Код товара": item,
        "Эластичность (E)": elasticity,
        "R-квадрат": r_squared,
        "Скидка по модели": discount_to_x2
    })

elasticity_df = pd.DataFrame(results).set_index("Код товара")


# сверим с фактом: рост продаж при наибольшей наблюдавшейся скидке, не превышающей скидку модели

pairs = elasticity_df.reset_index()[["Код товара", "Скидка по модели"]].merge(
    sales_by_discount[["Код товара", "Скидка_%", "Рост"]], on="Код товара"
)

pairs = pairs[pairs["Скидка_%"] <= pairs["Скидка по модели"] * 100]

pairs = pairs.sort_values("Скидка_%", ascending=False).drop_duplicates("Код товара").set_index("Код товара")

elasticity_df["Скидка в данных, %"] = pairs["Скидка_%"]
elasticity_df["Факт. рост"] = pairs["Рост"]

print("Эластичность и скидка по модели + сверка с фактом")
display(elasticity_df.style.format({
    "Эластичность (E)": "{:.2f}",
    "R-квадрат": "{:.2f}",
    "Скидка по модели": "{:.1%}",
    "Скидка в данных, %": "{:.0f}",
    "Факт. рост": "{:.1f}x"
}, na_rep="нет наблюдений ниже 25%"))

Эластичность и скидка по модели + сверка с фактом


,Эластичность (E),R-квадрат,Скидка по модели,"Скидка в данных, %",Факт. рост
Код товара,,,,,
3226300004,-2.58,0.59,23.5%,20,1.5x
15111300017,-3.70,0.33,17.1%,15,1.1x
15111300006,-3.21,0.54,19.4%,10,1.7x
15110900016,-2.55,0.40,23.8%,20,1.2x
19760336592,-2.66,0.42,23.0%,20,1.4x
14011400047,-2.71,0.66,22.5%,нет наблюдений ниже 25%,нет наблюдений ниже 25%
14012900005,-3.09,0.65,20.1%,15,1.7x
19000013047,-2.83,0.65,21.7%,нет наблюдений ниже 25%,нет наблюдений ниже 25%
3234800007,-2.94,0.48,21.0%,20,1.6x
